In [1]:
import torch
import matplotlib.pyplot as plt
import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)
from pathlib import Path
import torch
import numpy as np
from tqdm.auto import tqdm

from src.io import load_dataset
from src.dataset import OCTLayerDataset
from src.models.unet import UNet
from src.training.split import split_subjects

/Users/philipabakah/Desktop/projects/explore/oct-retinal-layer-segmentation


In [2]:
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

mps


In [4]:
model = UNet(
    n_channels=1,
    n_classes=8,
)

model.load_state_dict(
    torch.load(
        "../checkpoints/best_model.pth",
        map_location=device,
    )
)

model.to(device)

model.eval()

UNet(
  (inc): DoubleConv(
    (block): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): DownBlock(
    (block): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): DoubleConv(
        (block): Sequential(
          (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)

In [5]:
subjects = load_dataset("../datasets/duke/2015_BOE_Chiu")

_, val_subjects = split_subjects(subjects)

val_dataset = OCTLayerDataset(
    val_subjects,
    grader=1,
)

print(len(val_dataset))

22


In [6]:
def dice_score(prediction, target, num_classes=8):

    scores = []

    for c in range(num_classes):

        pred = prediction == c
        gt = target == c

        intersection = (pred & gt).sum()

        union = pred.sum() + gt.sum()

        if union == 0:
            scores.append(1.0)
            continue

        dice = (2.0 * intersection) / union

        scores.append(float(dice))

    return scores

In [7]:
def iou_score(prediction, target, num_classes=8):

    scores = []

    for c in range(num_classes):

        pred = prediction == c
        gt = target == c

        intersection = (pred & gt).sum()

        union = (pred | gt).sum()

        if union == 0:
            scores.append(1.0)
            continue

        scores.append(float(intersection / union))

    return scores

In [8]:
def pixel_accuracy(prediction, target):

    return float(
        (prediction == target).sum()
        / target.size
    )

In [9]:
all_dice = []
all_iou = []
all_accuracy = []

with torch.no_grad():

    for image, mask in tqdm(val_dataset):

        output = model(
            image.unsqueeze(0).to(device)
        )

        prediction = (
            output.argmax(dim=1)
            .squeeze(0)
            .cpu()
            .numpy()
        )

        mask = mask.numpy()

        all_dice.append(
            dice_score(prediction, mask)
        )

        all_iou.append(
            iou_score(prediction, mask)
        )

        all_accuracy.append(
            pixel_accuracy(
                prediction,
                mask,
            )
        )

  0%|          | 0/22 [00:00<?, ?it/s]

In [10]:
all_dice = np.array(all_dice)

all_iou = np.array(all_iou)

all_accuracy = np.array(all_accuracy)

In [11]:
print("=" * 40)

print(
    f"Mean Dice : {all_dice.mean():.4f}"
)

print(
    f"Mean IoU  : {all_iou.mean():.4f}"
)

print(
    f"Pixel Acc : {all_accuracy.mean():.4f}"
)

print("=" * 40)

Mean Dice : 0.8328
Mean IoU  : 0.7233
Pixel Acc : 0.9711


In [12]:
layer_names = [
    "Layer 0",
    "Layer 1",
    "Layer 2",
    "Layer 3",
    "Layer 4",
    "Layer 5",
    "Layer 6",
    "Layer 7",
]

for i, name in enumerate(layer_names):

    print(
        f"{name:10s}: {all_dice[:, i].mean():.4f}"
    )

Layer 0   : 0.9899
Layer 1   : 0.8001
Layer 2   : 0.8870
Layer 3   : 0.7587
Layer 4   : 0.7282
Layer 5   : 0.8603
Layer 6   : 0.8299
Layer 7   : 0.8085


In [13]:
mean_dice_per_scan = all_dice.mean(axis=1)

worst = np.argsort(mean_dice_per_scan)[:10]

print(worst)

[11 18  4 21  0  3  2 16 20 14]


In [14]:
best = np.argsort(mean_dice_per_scan)[-10:]

print(best)

[17  6  9  1  8 19  7 10 13 12]
